# Python Type Hints

You will learn:
- Why type hints are useful
- Basic syntax for functions, variables, and classes
- Typing collections, optionals, unions
- Type aliases, `TypedDict`, generics, and `Callable`
- A few advanced tools such as `Literal`, `Final`, `ClassVar`, and `NewType`
- How to use type checkers like `mypy` / `pyright`

There are also small exercises at the end of some sections.

---
## 0. Prerequisites

You should already be comfortable with:
- Basic Python syntax
- Functions and classes
- Lists, dicts, sets, tuples

You do **not** need prior experience with type hints.

## 1. Why Type Hints?

Type hints in Python:

- **Do not** change how the code runs (no runtime checks by default).
- **Do** help:
  - Static analyzers: `mypy`, `pyright`, `ruff`.
  - IDEs (VS Code, PyCharm) with better autocomplete and refactorings.
  - Humans reading the code (documentation of intent).

Example **without** hints:
```python
def add(a, b):
    return a + b
```

With hints:
```python
def add(a: int, b: int) -> int:
    return a + b
```

Now a type checker can catch `add("x", "y")` as an error.


In [ ]:
def add(a: int, b: int) -> int:
    return a + b

print(add(1, 2))          # OK
print(add(10, -5))       # OK

# A static type checker will complain about this:
# add("x", "y")


---
## 2. Basic Syntax

### 2.1 Function Annotations

General pattern:

```python
def func(param1: Type1, param2: Type2) -> ReturnType:
    ...
```

Example:


In [ ]:
def greet(name: str, repeat: int) -> str:
    return ("Hello " + name + "! ") * repeat

print(greet("Alice", 2))
# A type checker would warn on: greet(123, "x")


### 2.2 Variable Annotations

You can annotate variables at module, class, or function scope:

```python
age: int = 20
name: str = "Alice"
pi: float = 3.14159
flag: bool = True
```

This helps readers and tools understand expected types.


In [ ]:
age: int = 20
name: str = "Alice"
pi: float = 3.14159
flag: bool = True

print(age, name, pi, flag)


---
## 3. Collections

In modern Python (3.9+), you can use the **built-in** generics for collections:

- `list[int]`, `dict[str, int]`, `set[str]`, `tuple[int, int]`, etc.

Older style (pre‑3.9) used `typing.List`, `typing.Dict`, etc. Both are still valid, but the built-in style is recommended.

### 3.1 Basic Collections

Examples using Python 3.9+ style:


In [ ]:
numbers: list[int] = [1, 2, 3]
names: list[str] = ["Alice", "Bob"]

unique_ids: set[int] = {1, 2, 3}

scores: dict[str, float] = {"Alice": 15.5, "Bob": 18.0}

point: tuple[int, int] = (10, 20)

print(numbers)
print(names)
print(unique_ids)
print(scores)
print(point)


### 3.2 Heterogeneous Tuples

Tuples can have fixed positions with different types:

```python
record: tuple[int, str, float] = (1, "Alice", 18.5)
```

This means position 0 is `int`, position 1 is `str`, position 2 is `float`.


In [ ]:
record: tuple[int, str, float] = (1, "Alice", 18.5)
print(record)
print("id:", record[0], "name:", record[1], "score:", record[2])


---
## 4. `Optional` and `Union`

Sometimes a value can be of multiple types.

### 4.1 `Optional`

    
`Optional[T]` is equivalent to `T | None` (or `Union[T, None]`). It means the value can be `T` or `None`.

Python 3.10+ syntax uses `|`.


In [ ]:
from typing import Optional

def find_user(id: int) -> Optional[str]:  # same as str | None in Python 3.10+
    if id == 1:
        return "Alice"
    return None

print(find_user(1))   # "Alice"
print(find_user(2))   # None


### 4.2 `Union` / `|`

A value can be of one **among several types**.

Older style:
```python
from typing import Union
Number = Union[int, float]
```

Modern shorthand (3.10+):
```python
Number = int | float
```


In [4]:
Number = int | float

def area(radius: Number) -> float:
    return 3.14 * float(radius) ** 2

print(area(3))     # int
print(area(2.5))   # float

28.26
19.625


---
## 5. Type Aliases

Type aliases give a **name** to complex types, improving readability.

Example:

```python
UserId = int
Score = float
UserScores = dict[UserId, list[Score]]
```

Now `UserScores` is easier to reuse and understand.


In [3]:
UserId = int
Score = float
UserScores = dict[UserId, list[Score]]

def get_average(scores: UserScores, user_id: UserId) -> float:
    user_scores = scores.get(user_id, [])
    return sum(user_scores) / len(user_scores) if user_scores else 0.0

data: UserScores = {
    1: [10.0, 20.0, 15.0],
    2: [18.0],
}

print(get_average(data, 1))
print(get_average(data, 2))
print(get_average(data, 3))

15.0
18.0
0.0


---
## 6. `TypedDict` – Typed Dictionaries (JSON‑like)

`TypedDict` lets you describe the expected **keys and value types** of a dictionary, often used for JSON‑like structures.

At runtime, you still get a normal `dict`, but type checkers understand its shape.


In [ ]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    active: bool

def deactivate(user: User) -> None:
    user["active"] = False

u: User = {"id": 1, "name": "Alice", "active": True}
print("Before:", u)
deactivate(u)
print("After: ", u)


---
## 7. Classes and Methods

You can annotate attributes and methods for better clarity and tooling.

### 7.1 Instance Attributes and Methods


In [ ]:
class Point:
    x: float
    y: float

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def distance_to_origin(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = Point(3.0, 4.0)
print(p.distance_to_origin())  # 5.0


### 7.2 `self`, `cls` and Forward References

- We usually **do not** annotate `self` or `cls` parameters.
- But we annotate the rest.
- For types not yet defined (e.g., the class itself), we can use **strings**.


In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2

    @classmethod
    def unit(cls) -> "Circle":  # forward reference as string
        return cls(1.0)

c = Circle.unit()
print(c.radius, c.area())


---
## 8. `Any` vs `object`

### 8.1 `Any`

`Any` means: "turn off type checking for this value".

Anything can be:
- assigned to `Any`
- passed where `Any` is expected
- any method/attribute can be accessed on `Any` without error from the checker.

Use it sparingly; it hides type bugs.


In [ ]:
from typing import Any

def process(value: Any) -> Any:
    # Type checker cannot help much here
    print("Processing:", value)
    return value

process(10)
process("hello")
process([1, 2, 3])


### 8.2 `object`

`object` is the base class of all types, but not as permissive as `Any`.

You can pass any value to something typed as `object`, but you cannot call arbitrary methods on it without casting or checking.


In [ ]:
def log(value: object) -> None:
    print("LOG:", value)

log(10)
log("hello")
log([1, 2, 3])

# value.upper() would be invalid here because value is only 'object'


---
## 9. Generics – Parameterized Functions and Classes

Generics let you write code that works for **any type**, while preserving type information.

We use `TypeVar` and `Generic`.


### 9.1 Generic Functions

Example: a function returning the first element of any iterable, preserving its element type.


In [7]:
from collections.abc import Iterable
from typing import TypeVar

T = TypeVar("T")

def head(items: Iterable[T]) -> T | None:
    for x in items:
        return x
    return None

print(head([1, 2, 3]))     # int | None
print(head(["a", "b"]))  # str | None
print(head([]))            # None

1
a
None


### 9.2 Generic Classes

Example: a `Box` that holds any type but keeps track of what it holds.


In [ ]:
from typing import Generic

T = TypeVar("T")

class Box(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value

    def get(self) -> T:
        return self.value

    def set(self, value: T) -> None:
        self.value = value

int_box = Box[int](10)
str_box = Box[str]("hello")

print(int_box.get() + 5)
print(str_box.get().upper())


---
## 10. `Callable` – Typing Functions as Values

`Callable[[ArgType1, ArgType2], ReturnType]` describes the **signature** of a function.

Example: a function `apply` that takes a binary operation.


In [ ]:
from typing import Callable

Op = Callable[[int, int], int]

def apply(a: int, b: int, op: Op) -> int:
    return op(a, b)

def add_op(x: int, y: int) -> int:
    return x + y

def mul_op(x: int, y: int) -> int:
    return x * y

print(apply(2, 3, add_op))
print(apply(2, 3, mul_op))


If you don't care about the parameter types:

```python
Callable[..., Any]
```

means: any parameters, any return type.


In [ ]:
from typing import Any

Fn = Callable[..., Any]

def call_twice(fn: Fn, *args: Any, **kwargs: Any) -> tuple[Any, Any]:
    return fn(*args, **kwargs), fn(*args, **kwargs)

def echo(x: str) -> str:
    return x

print(call_twice(echo, "hello"))


---
## 11. Literal, Final, ClassVar, NewType

These are more advanced but useful in real projects.

### 11.1 `Literal`

`Literal` restricts a value to specific constants.

```python
Status = Literal["open", "closed", "pending"]
```


In [ ]:
from typing import Literal

Status = Literal["open", "closed", "pending"]

def set_status(s: Status) -> None:
    print("Setting status to", s)

set_status("open")
set_status("closed")
# set_status("unknown")  # type checker would warn


### 11.2 `Final`

`Final` marks variables (and in some cases methods/classes) as not meant to be changed or overridden.

Checkers will warn if you reassign a `Final` variable.


In [ ]:
from typing import Final

MAX_SIZE: Final = 100
print(MAX_SIZE)

# Reassignment would be flagged by a type checker:
# MAX_SIZE = 200


### 11.3 `ClassVar`

`ClassVar` marks attributes that are class-level, not instance-level (useful for dataclasses and clarity).


In [ ]:
from typing import ClassVar

class Config:
    VERSION: ClassVar[str] = "1.0"

    def __init__(self, name: str) -> None:
        self.name = name

cfg = Config("App")
print(cfg.name, Config.VERSION)


### 11.4 `NewType`

`NewType` creates a **distinct type** based on an existing one.

This helps avoid mixing different IDs accidentally (e.g., `UserId` vs `OrderId`).


In [ ]:
from typing import NewType

UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)

def get_user_name(user_id: UserId) -> str:
    return f"User#{int(user_id)}"

uid = UserId(10)
oid = OrderId(10)

print(get_user_name(uid))
# get_user_name(oid)  # static checker will warn: expected UserId, got OrderId


---
## 12. Type Checking in Practice (mypy / pyright)

Type hints are most useful when combined with **static type checkers**.

### 12.1 mypy

Install:
```bash
pip install mypy
```

Run:
```bash
mypy your_file.py
```

Example file `example.py`:
```python
def add(a: int, b: int) -> int:
    return a + b

add(1, 2)      # OK
add("x", "y")  # mypy error
```

### 12.2 pyright

- Available as VS Code extension, or CLI (`pyright`).
- Often very fast, great editor integration.

In VS Code, just install the Python/pyright support and it will highlight type issues in real time.

---
## 13. Gradual Typing Strategy for Real Projects

You don't have to type everything at once.

1. **Start with new code**: add type hints to new modules and functions.
2. **Add hints to critical areas**: business logic, APIs.
3. **Use `Any` as a temporary escape hatch** for very dynamic or legacy parts.
4. **Enable type checking in CI** to prevent regressions.
5. Gradually tighten rules as your code base becomes more typed.

---
# 14. Practice Exercises

Use the following cells to practice. Try to run a type checker (`mypy`) on your file/notebook to see if your types are correct.

### Exercise 1 – Basic Function Types

Add type hints to this function:

```python
def repeat(text, times):
    return text * times
```

- `text` should be `str`.
- `times` should be `int`.
- Return type is `str`.

Then call `repeat("hi", 3)` and `repeat(123, 3)` and see what a type checker says.


In [ ]:
# TODO: Exercise 1

# 1. Add proper type hints
def repeat(text, times):
    return text * times

# 2. Test calls
print(repeat("hi", 3))
print(repeat(123, 3))  # static type checker should warn about this


### Exercise 2 – Collections

Annotate the function below:

```python
def average(values):
    return sum(values) / len(values)
```

- `values` should be something like `list[int]` or `list[int | float]`.
- Return type is `float`.

Test with `average([1, 2, 3])` and `average([1.0, 2.5, 3.0])`.

Then try `average(["a", "b"])` and see what your type checker says.


In [8]:
# TODO: Exercise 2
# Add type hints to `average`, then test it.

def average(values):
    return sum(values) / len(values)

print(average([1, 2, 3]))
print(average([1.0, 2.5, 3.0]))

# average(["a", "b"])  # would fail at runtime; a type checker warns first

2.0
2.1666666666666665


### Exercise 3 – Optional and Union

Implement and annotate:

```python
def parse_int(value):
    ...
```

- Input: `str | int`
- Output: `int | None`

Behavior:
- If `value` is `int`, return it as is.
- If `value` is `str` and can be converted with `int(value)`, return the result.
- Otherwise return `None`.


In [ ]:
# TODO: Exercise 3

def parse_int(value):
    # Add type hints and implementation
    pass

print(parse_int(10))
print(parse_int("20"))
print(parse_int("abc"))


### Exercise 4 – Generic Box

Implement a generic class:

```python
T = TypeVar("T")

class Box(Generic[T]):
    ...
```

Requirements:
- `__init__(value: T) -> None`
- `get() -> T`
- `set(value: T) -> None`

Create `box_int = Box[int](123)` and `box_str = Box[str]("abc")` and use them.


In [9]:
# TODO: Exercise 4
from typing import Generic, TypeVar

T = TypeVar("T")

class Box(Generic[T]):
    ...  # implement __init__, get, and set

# After implementing, uncomment these lines:
# box_int = Box[int](123)
# box_str = Box[str]("abc")
# print(box_int.get() + 1)
# print(box_str.get().upper())

---
# 15. Summary

In this notebook, you learned:

- Basic syntax for type hints on functions and variables
- Typing collections: `list`, `dict`, `tuple`, etc.
- Using `Optional`, `Union`, and type aliases
- Structuring data with `TypedDict`
- Typing classes, methods, and attributes
- `Any` vs `object` and when to use them
- Generics with `TypeVar` and `Generic`
- Function types with `Callable`
- Advanced tools: `Literal`, `Final`, `ClassVar`, `NewType`
- How to use `mypy` / `pyright` for static checking

You can extend this notebook by:
- Adding your own functions and classes and annotating them
- Running a type checker regularly to see real benefits
- Combining type hints with unit tests for strong guarantees